# GraphRAG Evaluation

Run every question in `graphrag/evaluation/ground_truth.json` through the pipeline,
score each answer with an LLM judge, and report two headline numbers:

- **Doc recall** — fraction of expected source files the pipeline retrieved.
- **Correctness (1–5)** — how well the system answer matches the ground-truth answer,
  scored by the LLM judge.

The conversation history is reset between categories, so each category is evaluated
as its own independent chat session.

**Prerequisite:** the main pipeline notebook (`GraphRAG main.ipynb`) has been run, so
Neo4j contains the documents, chunks, entities, and communities.

In [ ]:
import graphrag.config as config
from graphrag.connections import init_connections

driver, emb_model, llm = init_connections(config)

In [ ]:
from graphrag.evaluation.runner import run_evaluation

GROUND_TRUTH_PATH = "graphrag/evaluation/ground_truth.json"

# Force a fresh chat history before these questions, on top of the
# category-boundary reset, because the follow-up classifier sometimes
# mis-labels them as follow-ups and tanks their retrieval.
HISTORY_RESET_IDS = ["Q009", "Q010", "Q011"]

df = run_evaluation(
    ground_truth_path=GROUND_TRUTH_PATH,
    driver=driver,
    embed_fn=emb_model.embed_query,
    llm=llm,
    cfg=config,
    verbose=True,
    history_reset_ids=HISTORY_RESET_IDS,
)

In [ ]:
from graphrag.evaluation.runner import summarize

stats = summarize(df)

print("=" * 50)
print("  HEADLINE RESULTS")
print("=" * 50)
print(f"  Correctness (LLM judge):  {stats['overall_correctness']:.2f} / 5")
if stats["overall_doc_recall"] is not None:
    print(f"  Document recall:          {stats['overall_doc_recall']:.2f}")
print(f"  Questions evaluated:      {len(df)}")
print("=" * 50)

In [ ]:
import pandas as pd

# Per-category breakdown: mean correctness and doc_recall per category.
category_summary = (
    df.groupby("category")
      .agg(n=("question_id", "count"),
           correctness=("correctness", "mean"),
           doc_recall=("doc_recall", "mean"))
      .round(2)
)
category_summary

In [ ]:
# Per-question detail. Wide wrapping for answer + judge reasoning;
# narrow centered columns for the IDs and numbers.
view = df[["question_id", "category", "doc_recall", "correctness",
           "answer", "judge_reasoning", "latency_s"]]

view.style.set_properties(
    subset=["answer", "judge_reasoning"],
    **{"width": "500px", "white-space": "pre-wrap", "text-align": "left",
       "vertical-align": "top"},
).set_properties(
    subset=["question_id", "category", "doc_recall", "correctness", "latency_s"],
    **{"width": "70px", "text-align": "center", "vertical-align": "top"},
)

In [ ]:
# Inspect the full system answer for one question. Change qid to dig into any row.
qid = "Q002"

row = df[df["question_id"] == qid].iloc[0]
print(f"Q ({row['category']}): {row['question']}\n")
print(f"-- System answer --\n{row['answer']}\n")
print(f"-- Judge reasoning --\n{row['judge_reasoning']}\n")
print(f"doc_recall={row['doc_recall']}  correctness={row['correctness']}  latency={row['latency_s']}s")